# DSCI 519 · Homework 1
## Prompt Engineering for SQL on an Unfamiliar Database

**Covers:** Session 1 (Chapters 1 to 3). **Points:** 100. **Submit:** this notebook (`.ipynb`) with all outputs, plus the PDF
made by the last cell of this notebook (not File > Print, which cuts long notebooks off).

### The situation
You are the new analyst at **Prairie Outfitters**, a fictional outdoor retailer with five stores in the Upper Midwest and an
online shop. Leadership has questions; the answers are in a SQLite database that no AI model has ever seen. Your job is to
use an AI companion effectively **and** prove that the answers are right.

### Rules for AI use in this course
* You may use any AI assistant or API. Using AI is the point of the course.
* You must **log** what you asked (prompt log), which provider and model answered, and how you **verified** the result.
* You are responsible for every number you report. "The AI said so" is never a verification.

### Grading rubric
| Part | What | Points |
|---|---|---|
| A | Setup, schema card, and data dictionary context | 10 |
| B | Context experiment (no schema vs schema vs schema + dictionary) | 20 |
| C | Six business questions with assigned techniques, verified | 42 (7 each) |
| D | Failure log: at least three documented AI failures and fixes | 15 |
| E | Reflection memo | 13 |

For each question in Part C, full credit requires: the technique was actually used (prompt shown), the final SQL runs and
answers the business question as defined in the data dictionary, and at least one **independent verification** is shown
(a second query, a row-count check, a hand-checked example, or a self-consistency comparison).

In [ ]:
#@title Setup: install packages and the course helper (run me first)
# Installs the vendor SDKs (quietly) and writes the course helper module llmkit.py
# into the Colab working folder. Run this cell once per Colab session.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai", "anthropic", "google-genai", "pydantic"], check=False)
LLMKIT_SOURCE = r'''"""
llmkit: a small, vendor-agnostic LLM client for DSCI 519 (Advanced Business Analytics).

Why this file exists
--------------------
Every AI vendor ships its own Python SDK, and each SDK shapes requests and
responses a little differently. OpenAI returns `choices[0].message.content`,
Anthropic returns a list of `content` blocks, Google returns `.text`, and the
token counters all have different names. If we wrote our pipelines directly
against one SDK, switching vendors would mean rewriting every pipeline.

This module is an *adapter*: one small interface (`LLM.chat`, `LLM.ask`,
`LLM.extract`) on top of several providers. Your pipeline code talks to the
adapter; the adapter talks to the vendor. That is the same design idea that
data engineers use for database drivers (one SQL interface, many databases).

Supported providers
-------------------
    "openai"      OpenAI API                 (key: OPENAI_API_KEY)
    "anthropic"   Anthropic Claude API       (key: ANTHROPIC_API_KEY)
    "google"      Google Gemini API          (key: GOOGLE_API_KEY or GEMINI_API_KEY)
    "groq"        Groq (OpenAI-compatible)   (key: GROQ_API_KEY)        free tier
    "openrouter"  OpenRouter (OpenAI-compat) (key: OPENROUTER_API_KEY)  some free models
    "mistral"     Mistral (OpenAI-compat)    (key: MISTRAL_API_KEY)
    "ollama"      Local models via Ollama    (no key; needs a local server)
    "compatible"  Any other OpenAI-compatible endpoint (pass base_url and key_name)
    "mock"        Offline fake model for testing pipeline wiring (no key, no cost)

Model names change often. The defaults below were checked in October 2026.
If a call fails with "model not found", run `LLM(provider).list_models()`
and pick a current model from your provider's list.

Quick start
-----------
    from llmkit import LLM
    llm = LLM("groq")                              # or "openai", "anthropic", "google", ...
    print(llm.ask("Say hello in five words."))

    from pydantic import BaseModel
    class Sentiment(BaseModel):
        label: str
        score: float
    result = llm.extract("I love this product!", Sentiment,
                         instructions="Classify the sentiment of the review.")
    print(result.label, result.score)

    llm.usage()                                     # calls, tokens, latency, cost
"""

from __future__ import annotations

import hashlib
import json
import os
import re
import time
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional, Type

__version__ = "1.1.0"

# ---------------------------------------------------------------------------
# 1. Provider registry: the only place where vendor-specific settings live.
# ---------------------------------------------------------------------------
PROVIDERS: Dict[str, Dict[str, Any]] = {
    "openai": {
        "style": "openai",
        "key_names": ["OPENAI_API_KEY"],
        "base_url": None,
        "default_model": "gpt-6-luna",
    },
    "anthropic": {
        "style": "anthropic",
        "key_names": ["ANTHROPIC_API_KEY"],
        "base_url": None,
        "default_model": "claude-haiku-4-5",
    },
    "google": {
        "style": "google",
        "key_names": ["GOOGLE_API_KEY", "GEMINI_API_KEY"],
        "base_url": None,
        "default_model": "gemini-3.5-flash-lite",
    },
    "groq": {
        "style": "compatible",
        "key_names": ["GROQ_API_KEY"],
        "base_url": "https://api.groq.com/openai/v1",
        "default_model": "llama-3.3-70b-versatile",
    },
    "openrouter": {
        "style": "compatible",
        "key_names": ["OPENROUTER_API_KEY"],
        "base_url": "https://openrouter.ai/api/v1",
        "default_model": "openai/gpt-oss-20b:free",
    },
    "mistral": {
        "style": "compatible",
        "key_names": ["MISTRAL_API_KEY"],
        "base_url": "https://api.mistral.ai/v1",
        "default_model": "mistral-small-latest",
    },
    "ollama": {
        "style": "compatible",
        "key_names": [],
        "base_url": "http://localhost:11434/v1",
        "default_model": "llama3.2",
    },
    "compatible": {
        "style": "compatible",
        "key_names": [],
        "base_url": None,
        "default_model": None,
    },
    "mock": {
        "style": "mock",
        "key_names": [],
        "base_url": None,
        "default_model": "mock-1",
    },
}


# ---------------------------------------------------------------------------
# 2. Secrets: never paste API keys into notebook cells.
# ---------------------------------------------------------------------------
def get_secret(name: str, prompt_if_missing: bool = True) -> Optional[str]:
    """Look up a secret in (1) Colab Secrets, (2) environment variables,
    (3) an interactive hidden prompt. Returns None if nothing is found."""
    # (1) Google Colab Secrets panel (the key icon in the left sidebar)
    try:
        from google.colab import userdata  # type: ignore

        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass  # secret not defined or notebook access not granted
    except ImportError:
        pass  # not running in Colab
    # (2) Environment variable (local Jupyter, .env files, CI)
    value = os.environ.get(name)
    if value:
        return value
    # (3) Ask the user without echoing the key on screen
    if prompt_if_missing:
        try:
            from getpass import getpass

            value = getpass(f"Paste your {name} (input is hidden): ").strip()
            if value:
                os.environ[name] = value  # remember for this session only
                return value
        except Exception:
            pass
    return None


# ---------------------------------------------------------------------------
# 3. A normalized response object: the same shape for every vendor.
# ---------------------------------------------------------------------------
@dataclass
class LLMResponse:
    text: str
    provider: str
    model: str
    input_tokens: int = 0
    output_tokens: int = 0
    latency_s: float = 0.0
    finish_reason: Optional[str] = None
    cached: bool = False
    raw: Any = field(default=None, repr=False)

    @property
    def total_tokens(self) -> int:
        return (self.input_tokens or 0) + (self.output_tokens or 0)


class LLMError(RuntimeError):
    """Raised when a call cannot be completed (after retries)."""


class ExtractionError(LLMError):
    """Raised when the model's output never matched the schema."""

    def __init__(self, message: str, last_text: str = "", errors: Optional[List[str]] = None):
        super().__init__(message)
        self.last_text = last_text
        self.errors = errors or []


# ---------------------------------------------------------------------------
# 4. JSON helpers used for structured outputs.
# ---------------------------------------------------------------------------
_FENCE_RE = re.compile(r"^```(?:json|JSON)?\s*|\s*```$", re.MULTILINE)


def parse_json_object(text: str) -> Any:
    """Turn a model reply into a Python object. Tolerates markdown fences and
    short prose before or after the JSON (a common model habit)."""
    if text is None:
        raise ValueError("empty reply")
    cleaned = _FENCE_RE.sub("", text.strip()).strip()
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass
    # Fall back to the outermost {...} or [...] span.
    for open_ch, close_ch in (("{", "}"), ("[", "]")):
        start, end = cleaned.find(open_ch), cleaned.rfind(close_ch)
        if start != -1 and end > start:
            try:
                return json.loads(cleaned[start : end + 1])
            except json.JSONDecodeError:
                continue
    raise ValueError("reply is not valid JSON")


def schema_prompt(schema: Type[Any]) -> str:
    """The text we append to instructions so ANY model knows the exact shape
    we expect. Works with every vendor because it is just text."""
    schema_json = json.dumps(schema.model_json_schema(), indent=2)
    return (
        "Return ONLY one JSON object (no prose, no markdown code fences) that "
        "conforms to this JSON Schema:\n"
        f"{schema_json}\n"
        "Rules: use null when an optional value is not present in the input; "
        "never invent values; do not add keys that are not in the schema."
    )


def estimate_tokens(text: str) -> int:
    """Vendor-neutral rule of thumb: about 4 characters per token for English.
    Every vendor uses a different tokenizer, so treat this as an estimate."""
    return max(1, round(len(text or "") / 4))


# ---------------------------------------------------------------------------
# 5. The client.
# ---------------------------------------------------------------------------
class LLM:
    """One interface, many vendors.

    Parameters
    ----------
    provider : str            one of PROVIDERS (e.g., "groq", "openai", "google")
    model : str               model id; defaults to the provider's default_model
    api_key : str             optional; otherwise read with get_secret()
    base_url : str            only for "compatible"/"ollama" style endpoints
    key_name : str            secret name to read for a "compatible" provider
    temperature : float       default sampling temperature (None = provider default)
    max_tokens : int          default cap on output tokens
    min_interval : float      minimum seconds between calls (free-tier rate limits)
    max_retries : int         retries for rate limits, timeouts, server errors
    cache : bool              reuse identical past responses (saves money on reruns)
    cache_path : str          where cached responses are stored (JSON lines)
    price_per_million : tuple (input_usd, output_usd) per 1M tokens, for cost reports
    verbose : bool            print one line per call
    reasoning_headroom : int  if a reply is cut off by a small max_tokens (reasoning models
                              spend tokens on hidden "thinking"), retry once with this many
                              tokens and use it as the minimum from then on (0 = never)
    """

    def __init__(
        self,
        provider: str = "groq",
        model: Optional[str] = None,
        api_key: Optional[str] = None,
        base_url: Optional[str] = None,
        key_name: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: int = 2048,
        min_interval: float = 0.0,
        max_retries: int = 4,
        timeout: float = 90.0,
        cache: bool = False,
        cache_path: str = "llm_cache.jsonl",
        price_per_million: Optional[tuple] = None,
        verbose: bool = False,
        reasoning_headroom: int = 2048,
    ):
        provider = provider.lower().strip()
        if provider not in PROVIDERS:
            raise ValueError(f"Unknown provider '{provider}'. Choose one of: {sorted(PROVIDERS)}")
        cfg = PROVIDERS[provider]
        self.provider = provider
        self.style = cfg["style"]
        self.model = model or cfg["default_model"]
        if not self.model:
            raise ValueError("Please pass model=... for this provider.")
        self.base_url = base_url or cfg["base_url"]
        self.temperature = temperature
        self.max_tokens = max_tokens
        self.min_interval = min_interval
        self.max_retries = max_retries
        self.timeout = timeout
        self.price_per_million = price_per_million
        self.verbose = verbose
        self.reasoning_headroom = reasoning_headroom
        self._token_floor = 0  # raised automatically if the model needs room to "think"
        self.calls: List[Dict[str, Any]] = []  # one row per API call (for usage reports)
        self.warnings: List[str] = []
        self.last: Optional[LLMResponse] = None
        self._last_call_time = 0.0
        self._dropped_params: set = set()  # params a model rejected (e.g., temperature)
        self._mock_schema = None

        # API key (not needed for mock or local ollama)
        names = ([key_name] if key_name else []) + list(cfg["key_names"])
        self._api_key = api_key
        if self._api_key is None and self.style != "mock" and names:
            for i, name in enumerate(names):
                self._api_key = get_secret(name, prompt_if_missing=(i == len(names) - 1))
                if self._api_key:
                    break
            if not self._api_key:
                raise LLMError(
                    f"No API key found for '{provider}'. Add {names[0]} to Colab Secrets "
                    "(key icon in the left sidebar) and enable notebook access."
                )
        if provider == "ollama" and not self._api_key:
            self._api_key = "ollama"  # Ollama ignores the key but the SDK wants one

        # Response cache (optional)
        self.cache_enabled = cache
        self.cache_path = cache_path
        self._cache: Dict[str, Dict[str, Any]] = {}
        if cache and os.path.exists(cache_path):
            with open(cache_path, "r", encoding="utf-8") as fh:
                for line in fh:
                    try:
                        row = json.loads(line)
                        self._cache[row["key"]] = row
                    except Exception:
                        continue

        self._client = None  # created lazily on the first call

    def __repr__(self) -> str:
        return f"LLM(provider='{self.provider}', model='{self.model}')"

    # ---------------------------------------------------------------- clients
    def _get_client(self):
        if self._client is not None:
            return self._client
        if self.style in ("openai", "compatible"):
            from openai import OpenAI

            kwargs = {"api_key": self._api_key, "timeout": self.timeout, "max_retries": 0}
            if self.base_url:
                kwargs["base_url"] = self.base_url
            self._client = OpenAI(**kwargs)
        elif self.style == "anthropic":
            import anthropic

            kwargs = {"api_key": self._api_key, "timeout": self.timeout, "max_retries": 0}
            if self.base_url:
                kwargs["base_url"] = self.base_url
            self._client = anthropic.Anthropic(**kwargs)
        elif self.style == "google":
            from google import genai

            kwargs = {"api_key": self._api_key}
            if self.base_url:
                from google.genai import types

                kwargs["http_options"] = types.HttpOptions(base_url=self.base_url)
            self._client = genai.Client(**kwargs)
        else:
            self._client = "mock"
        return self._client

    # ------------------------------------------------------- vendor adapters
    def _call_openai_style(self, system, messages, temperature, max_tokens, json_mode):
        client = self._get_client()
        msgs = ([{"role": "system", "content": system}] if system else []) + messages
        kwargs: Dict[str, Any] = {"model": self.model, "messages": msgs}
        # OpenAI's newer models use max_completion_tokens; most compatible
        # servers still use max_tokens.
        if self.style == "openai":
            kwargs["max_completion_tokens"] = max_tokens
        else:
            kwargs["max_tokens"] = max_tokens
        if temperature is not None and "temperature" not in self._dropped_params:
            kwargs["temperature"] = temperature
        if json_mode and "response_format" not in self._dropped_params:
            kwargs["response_format"] = {"type": "json_object"}
        resp = client.chat.completions.create(**kwargs)
        choice = resp.choices[0]
        usage = getattr(resp, "usage", None)
        return LLMResponse(
            text=(choice.message.content or ""),
            provider=self.provider,
            model=getattr(resp, "model", self.model) or self.model,
            input_tokens=getattr(usage, "prompt_tokens", 0) or 0,
            output_tokens=getattr(usage, "completion_tokens", 0) or 0,
            finish_reason=getattr(choice, "finish_reason", None),
            raw=resp,
        )

    def _call_anthropic(self, system, messages, temperature, max_tokens, json_mode):
        client = self._get_client()
        kwargs: Dict[str, Any] = {"model": self.model, "max_tokens": max_tokens, "messages": messages}
        if system:
            kwargs["system"] = system
        if temperature is not None and "temperature" not in self._dropped_params:
            kwargs["extra_body"] = {"temperature": temperature}
        # Anthropic has no "JSON mode" flag; the schema prompt does the work.
        resp = client.messages.create(**kwargs)
        text = "".join(getattr(b, "text", "") for b in resp.content if getattr(b, "type", "") == "text")
        usage = getattr(resp, "usage", None)
        return LLMResponse(
            text=text,
            provider=self.provider,
            model=getattr(resp, "model", self.model) or self.model,
            input_tokens=getattr(usage, "input_tokens", 0) or 0,
            output_tokens=getattr(usage, "output_tokens", 0) or 0,
            finish_reason=getattr(resp, "stop_reason", None),
            raw=resp,
        )

    def _call_google(self, system, messages, temperature, max_tokens, json_mode):
        from google.genai import types

        client = self._get_client()
        contents = [
            types.Content(
                role=("model" if m["role"] == "assistant" else "user"),
                parts=[types.Part(text=m["content"])],
            )
            for m in messages
        ]
        cfg: Dict[str, Any] = {
            "max_output_tokens": max_tokens,
            # we never pass tools, so switch off automatic function calling
            "automatic_function_calling": types.AutomaticFunctionCallingConfig(disable=True),
        }
        if system:
            cfg["system_instruction"] = system
        if temperature is not None and "temperature" not in self._dropped_params:
            cfg["temperature"] = temperature
        if json_mode and "response_format" not in self._dropped_params:
            cfg["response_mime_type"] = "application/json"
        resp = client.models.generate_content(
            model=self.model, contents=contents, config=types.GenerateContentConfig(**cfg)
        )
        usage = getattr(resp, "usage_metadata", None)
        finish = None
        try:
            finish = str(resp.candidates[0].finish_reason)
        except Exception:
            pass
        return LLMResponse(
            text=(resp.text or ""),
            provider=self.provider,
            model=self.model,
            input_tokens=getattr(usage, "prompt_token_count", 0) or 0,
            output_tokens=(getattr(usage, "candidates_token_count", 0) or 0)
            + (getattr(usage, "thoughts_token_count", 0) or 0),
            finish_reason=finish,
            raw=resp,
        )

    def _call_mock(self, system, messages, temperature, max_tokens, json_mode):
        """A fake model. It never 'understands' anything: it returns a fixed
        reply, or (for structured calls) the simplest object that fits the
        schema. Use it to test that your pipeline code runs end to end."""
        prompt_text = (system or "") + " ".join(m["content"] for m in messages)
        if json_mode and self._mock_schema is not None:
            text = json.dumps(_mock_instance(self._mock_schema.model_json_schema()))
        elif json_mode:
            text = "{}"
        else:
            text = "MOCK RESPONSE (provider='mock' returns placeholder text)."
        return LLMResponse(
            text=text,
            provider="mock",
            model=self.model,
            input_tokens=estimate_tokens(prompt_text),
            output_tokens=estimate_tokens(text),
            finish_reason="stop",
        )

    # --------------------------------------------------------- error policy
    @staticmethod
    def _status_code(exc: Exception) -> Optional[int]:
        for attr in ("status_code", "code", "status"):
            val = getattr(exc, attr, None)
            if isinstance(val, int):
                return val
        resp = getattr(exc, "response", None)
        val = getattr(resp, "status_code", None)
        return val if isinstance(val, int) else None

    def _is_retryable(self, exc: Exception) -> bool:
        code = self._status_code(exc)
        name = type(exc).__name__.lower()
        msg = str(exc).lower()
        if code in (408, 409, 425, 429, 500, 502, 503, 504, 529):
            return True
        if any(w in name for w in ("ratelimit", "timeout", "connection", "overloaded", "unavailable")):
            return True
        return "resource_exhausted" in msg or "rate limit" in msg or "overloaded" in msg

    def _unsupported_param(self, exc: Exception) -> Optional[str]:
        """Some models reject parameters (e.g., reasoning models reject a custom
        temperature). Detect that so we can retry without the parameter."""
        code = self._status_code(exc)
        msg = str(exc).lower()
        if code not in (400, 422, None):
            return None
        for param in ("temperature", "response_format", "response_mime_type"):
            if param in msg and ("support" in msg or "unrecognized" in msg or "invalid" in msg or "not allowed" in msg):
                return "response_format" if param == "response_mime_type" else param
        return None

    # ------------------------------------------------------------- the call
    def chat(
        self,
        messages: List[Dict[str, str]],
        system: Optional[str] = None,
        temperature: Optional[float] = None,
        max_tokens: Optional[int] = None,
        json_mode: bool = False,
    ) -> LLMResponse:
        """Send a conversation and get one normalized LLMResponse back.

        messages: [{"role": "user" | "assistant", "content": "..."}, ...]
        system:   standing instructions (kept separate from messages because
                  vendors place them differently)
        """
        temperature = self.temperature if temperature is None else temperature
        max_tokens = max(max_tokens or self.max_tokens, self._token_floor)
        for m in messages:
            if m.get("role") not in ("user", "assistant"):
                raise ValueError("messages may only use roles 'user' and 'assistant'; pass system=... separately")

        # 1) Cache lookup: identical request -> identical (free) answer.
        key = None
        if self.cache_enabled:
            key = hashlib.sha256(
                json.dumps(
                    [self.provider, self.model, system, messages, temperature, max_tokens, json_mode],
                    sort_keys=True,
                ).encode("utf-8")
            ).hexdigest()
            if key in self._cache:
                row = self._cache[key]
                resp = LLMResponse(
                    text=row["text"],
                    provider=self.provider,
                    model=self.model,
                    input_tokens=row.get("input_tokens", 0),
                    output_tokens=row.get("output_tokens", 0),
                    cached=True,
                )
                self._log(resp)
                self.last = resp
                return resp

        adapter = {
            "openai": self._call_openai_style,
            "compatible": self._call_openai_style,
            "anthropic": self._call_anthropic,
            "google": self._call_google,
            "mock": self._call_mock,
        }[self.style]

        # 2) Call with polite pacing, retries, and parameter fallback.
        attempt = 0
        while True:
            wait = self.min_interval - (time.time() - self._last_call_time)
            if wait > 0:
                time.sleep(wait)
            start = time.time()
            self._last_call_time = start
            try:
                resp = adapter(system, messages, temperature, max_tokens, json_mode)
                resp.latency_s = round(time.time() - start, 3)
                break
            except Exception as exc:  # vendor SDKs raise different classes
                bad_param = self._unsupported_param(exc)
                if bad_param and bad_param not in self._dropped_params:
                    self._dropped_params.add(bad_param)
                    self._warn(f"Model '{self.model}' rejected '{bad_param}'; retrying without it.")
                    continue
                attempt += 1
                if attempt > self.max_retries or not self._is_retryable(exc):
                    raise LLMError(f"{self.provider} call failed: {type(exc).__name__}: {exc}") from exc
                delay = min(60.0, 2.0 ** attempt)
                self._warn(f"{type(exc).__name__} (attempt {attempt}); waiting {delay:.0f}s before retrying.")
                time.sleep(delay)

        cut_off = str(resp.finish_reason).lower() in ("length", "max_tokens", "finishreason.max_tokens")
        if (cut_off and (json_mode or not resp.text.strip()) and self.style != "mock"
                and self.reasoning_headroom and max_tokens < self.reasoning_headroom):
            # Typical of reasoning models: the hidden "thinking" used up the budget before the
            # answer was written. Retry once with more room and keep that minimum for later calls.
            self._log(resp)
            self._token_floor = self.reasoning_headroom
            self._warn(
                f"Reply cut off at max_tokens={max_tokens} (this model spends tokens on hidden reasoning). "
                f"Retrying with max_tokens={self.reasoning_headroom}; later calls from this client use at least that."
            )
            return self.chat(messages, system=system, temperature=temperature,
                             max_tokens=self.reasoning_headroom, json_mode=json_mode)
        if not resp.text.strip() and cut_off:
            self._warn(
                "The reply is empty because the token limit was reached (reasoning models spend "
                "tokens 'thinking'). Increase max_tokens."
            )

        # 3) Save to cache and usage log.
        if self.cache_enabled and key:
            row = {"key": key, "text": resp.text, "input_tokens": resp.input_tokens, "output_tokens": resp.output_tokens}
            self._cache[key] = row
            with open(self.cache_path, "a", encoding="utf-8") as fh:
                fh.write(json.dumps(row) + "\n")
        self._log(resp)
        self.last = resp
        return resp

    def ask(self, prompt: str, system: Optional[str] = None, **kwargs) -> str:
        """Single-turn convenience wrapper: prompt in, text out."""
        return self.chat([{"role": "user", "content": prompt}], system=system, **kwargs).text

    def extract(
        self,
        content: Any,
        schema: Type[Any],
        instructions: Optional[str] = None,
        max_repairs: int = 2,
        **kwargs,
    ):
        """Structured output that works with every vendor.

        1. Put the JSON Schema of `schema` (a Pydantic model) into the system prompt.
        2. Ask for JSON (using the vendor's JSON mode when it has one).
        3. Parse and validate the reply with Pydantic.
        4. If validation fails, show the model its own error and ask again
           (a "repair loop"), up to `max_repairs` times.
        Returns an instance of `schema`, or raises ExtractionError.
        """
        system = ((instructions.strip() + "\n\n") if instructions else "") + schema_prompt(schema)
        if not isinstance(content, str):
            content = json.dumps(content, default=str, ensure_ascii=False)
        messages = [{"role": "user", "content": content}]
        errors: List[str] = []
        text = ""
        self._mock_schema = schema
        try:
            for attempt in range(max_repairs + 1):
                resp = self.chat(messages, system=system, json_mode=True, **kwargs)
                text = resp.text
                try:
                    data = parse_json_object(text)
                    obj = schema.model_validate(data)
                    resp.attempts = attempt + 1  # type: ignore[attr-defined]
                    return obj
                except Exception as exc:  # JSON error or pydantic ValidationError
                    err = _short_error(exc)
                    errors.append(err)
                    messages = messages + [
                        {"role": "assistant", "content": text or "(empty reply)"},
                        {
                            "role": "user",
                            "content": "Your previous reply could not be used because: "
                            f"{err}\nReply again with ONLY the corrected JSON object.",
                        },
                    ]
        finally:
            self._mock_schema = None
        raise ExtractionError(
            f"No schema-valid reply after {max_repairs + 1} attempts. Last error: {errors[-1]}",
            last_text=text,
            errors=errors,
        )

    # ---------------------------------------------------------- bookkeeping
    def _warn(self, msg: str) -> None:
        self.warnings.append(msg)
        print(f"[llmkit] {msg}")

    def _cost(self, resp: LLMResponse) -> Optional[float]:
        if not self.price_per_million or resp.cached:
            return 0.0 if resp.cached else None
        p_in, p_out = self.price_per_million
        return (resp.input_tokens * p_in + resp.output_tokens * p_out) / 1_000_000

    def _log(self, resp: LLMResponse) -> None:
        row = {
            "time": time.strftime("%H:%M:%S"),
            "provider": resp.provider,
            "model": resp.model,
            "input_tokens": resp.input_tokens,
            "output_tokens": resp.output_tokens,
            "latency_s": resp.latency_s,
            "cached": resp.cached,
            "cost_usd": self._cost(resp),
        }
        self.calls.append(row)
        if self.verbose:
            print(
                f"[llmkit] {row['provider']}/{row['model']} in={row['input_tokens']} "
                f"out={row['output_tokens']} {row['latency_s']}s{' (cached)' if resp.cached else ''}"
            )

    def usage(self):
        """Summary of all calls made by this client (a pandas DataFrame if available)."""
        try:
            import pandas as pd

            df = pd.DataFrame(self.calls)
            if df.empty:
                print("No calls yet.")
                return df
            summary = {
                "calls": len(df),
                "cached_calls": int(df["cached"].sum()),
                "input_tokens": int(df["input_tokens"].sum()),
                "output_tokens": int(df["output_tokens"].sum()),
                "avg_latency_s": round(df.loc[~df["cached"], "latency_s"].mean(), 2) if (~df["cached"]).any() else 0.0,
                "est_cost_usd": (round(df["cost_usd"].sum(), 6) if df["cost_usd"].notna().any() else "set price_per_million"),
            }
            return pd.Series(summary, name=f"{self.provider}/{self.model}", dtype=object)
        except ImportError:
            return self.calls

    def reset_usage(self) -> None:
        self.calls = []

    def list_models(self) -> List[str]:
        """Ask the provider which models your key can use."""
        client = self._get_client()
        if self.style in ("openai", "compatible"):
            return sorted(m.id for m in client.models.list())
        if self.style == "anthropic":
            return sorted(m.id for m in client.models.list())
        if self.style == "google":
            return sorted(m.name.replace("models/", "") for m in client.models.list())
        return ["mock-1"]


# ---------------------------------------------------------------------------
# 6. Small private helpers.
# ---------------------------------------------------------------------------
def _short_error(exc: Exception) -> str:
    """Compact error text that a model (and a student) can read."""
    try:
        from pydantic import ValidationError

        if isinstance(exc, ValidationError):
            parts = []
            for e in exc.errors()[:6]:
                loc = ".".join(str(x) for x in e.get("loc", ())) or "(root)"
                parts.append(f"{loc}: {e.get('msg')}")
            return "; ".join(parts)
    except ImportError:
        pass
    return f"{type(exc).__name__}: {exc}"[:400]


def _mock_instance(schema: Dict[str, Any], defs: Optional[Dict[str, Any]] = None) -> Any:
    """Build the simplest value that satisfies a JSON Schema (used by provider='mock')."""
    defs = defs if defs is not None else schema.get("$defs", {})
    if "$ref" in schema:
        return _mock_instance(defs.get(schema["$ref"].split("/")[-1], {}), defs)
    if "enum" in schema:
        return schema["enum"][0]
    if "const" in schema:
        return schema["const"]
    for key in ("anyOf", "oneOf", "allOf"):
        if key in schema:
            options = [s for s in schema[key] if s.get("type") != "null"] or schema[key]
            return _mock_instance(options[0], defs)
    if "default" in schema and schema["default"] is not None:
        return schema["default"]
    typ = schema.get("type")
    if typ == "object" or "properties" in schema:
        props = schema.get("properties", {})
        return {k: _mock_instance(v, defs) for k, v in props.items()}
    if typ == "array":
        return []
    if typ == "integer":
        return int(schema.get("minimum", 0))
    if typ == "number":
        return float(schema.get("minimum", 0.0))
    if typ == "boolean":
        return False
    if typ == "null":
        return None
    return "mock"
'''
with open("llmkit.py", "w", encoding="utf-8") as fh:
    fh.write(LLMKIT_SOURCE)
import importlib, llmkit
importlib.reload(llmkit)
print("llmkit", llmkit.__version__, "is ready.")

### Choose your AI provider

Everything in this course works with any of the providers below. Pick one, make sure its API key is stored in
**Colab Secrets** (click the key icon in the left sidebar, add the secret name shown, and switch on
*Notebook access*), and run the cell. You never paste a key into the notebook itself.

| `PROVIDER` | Secret name | Cost notes (October 2026, verify before use) | Suggested `MIN_INTERVAL` |
|---|---|---|---|
| `"groq"` | `GROQ_API_KEY` | Free tier, no credit card, generous daily limits | 2 |
| `"google"` | `GOOGLE_API_KEY` | Free tier for Flash models (low daily quota; your data may be used for training) | 6 |
| `"openai"` | `OPENAI_API_KEY` | Paid, prepaid credit; small models cost cents for this course | 0 |
| `"anthropic"` | `ANTHROPIC_API_KEY` | Paid, prepaid credit; Haiku is the low-cost tier | 0 |
| `"openrouter"` | `OPENROUTER_API_KEY` | Many models behind one key; some `:free` models | 3 |
| `"mistral"` | `MISTRAL_API_KEY` | Free experimentation tier available | 2 |
| `"mock"` | none | Offline fake model: returns placeholders, useful only to test that code runs | 0 |

`MODEL = None` uses the provider's default from `llmkit.PROVIDERS`. Model names change frequently; if you get a
"model not found" error, run `llm.list_models()` and copy a current name into `MODEL`.

In [ ]:
from llmkit import LLM, get_secret, estimate_tokens

PROVIDER = "groq"      # "groq" | "google" | "openai" | "anthropic" | "openrouter" | "mistral" | "mock"
MODEL = None           # None = provider default, or a specific model id string
MIN_INTERVAL = 2.0     # seconds between calls (protects free-tier rate limits)

llm = LLM(PROVIDER, model=MODEL, min_interval=MIN_INTERVAL, cache=True)
print(llm)

In [ ]:
#@title Get the data files
# Course data: this cell makes the homework files available in /content/hw1.
# Option 1 (recommended): your instructor sets COURSE_DATA_URL to where the course data folder is hosted.
# Option 2: upload the files when prompted (download them from the course site first).
import os, urllib.request
COURSE_DATA_URL = ""   # e.g. "https://raw.githubusercontent.com/<account>/<repo>/main/data"
FOLDER = "hw1"
FILES = ['prairie_outfitters.db']
os.makedirs(FOLDER, exist_ok=True)
missing = [f for f in FILES if not os.path.exists(os.path.join(FOLDER, f))]
for f in list(missing):
    if COURSE_DATA_URL:
        try:
            urllib.request.urlretrieve(f"{COURSE_DATA_URL}/{FOLDER}/{f}", os.path.join(FOLDER, f))
            missing.remove(f)
        except Exception as exc:
            print("Download failed for", f, exc)
if missing:
    try:
        from google.colab import files as colab_files
        print("Please upload:", missing)
        uploaded = colab_files.upload()
        for name, content in uploaded.items():
            with open(os.path.join(FOLDER, name), "wb") as fh:
                fh.write(content)
    except ImportError:
        print("Not in Colab: copy these files into", FOLDER, ":", missing)
print("Files present:", sorted(os.listdir(FOLDER)))

---
# Part A. Setup and context (10 points)
1. Connect to the database and write `run_sql()` (you may copy it from Session 1).
2. Build a **schema card** with the `schema_card()` function from Session 1.
3. Read the data dictionary below. It contains business definitions that are *not* visible in the schema.

### Prairie Outfitters data dictionary (give this to your AI companion as context)
* **stores**: physical stores. `opened_date` is an ISO date.
* **customers**: `loyalty_tier` codes: `B` = Bronze, `S` = Silver, `G` = Gold, `P` = Platinum. Some customers never ordered.
* **products**: `list_price` is the catalog price; `is_active = 0` means discontinued.
* **orders**: one row per order. `channel` is `'online'` or `'in_store'`. **Online orders have `store_id = NULL`.**
  `status` is `'completed'` or `'cancelled'`; cancelled orders never count as revenue. `order_ts` is local time stored as
  text `YYYY-MM-DD HH:MM:SS`.
* **order_items**: one row per product line. The quantity column is **`qty`**. `unit_price` is the price charged before the
  line discount; `discount_pct` is a fraction (0.10 = 10% off).
  **Line revenue** = `qty * unit_price * (1 - discount_pct)`.
* **returns**: one row per returned line; `qty_returned` units, `refund_amount` dollars. Refunds are attributed to the
  period of the *original order*.
* **Net revenue** = line revenue of completed orders minus refunds on those lines.
* Data covers January 2024 through December 2025. The database is **SQLite**.

In [ ]:
import sqlite3, re, pandas as pd
con = sqlite3.connect("hw1/prairie_outfitters.db")

def run_sql(query, limit=30):
    df = pd.read_sql_query(query, con)
    return df.head(limit) if len(df) > limit else df

def schema_card(con):
    lines = []
    for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
        cols = con.execute(f"PRAGMA table_info({t})").fetchall()
        lines.append(f"{t}(" + ", ".join(f"{c[1]} {c[2]}" + (" PK" if c[5] else "") for c in cols) + ")")
        for fk in con.execute(f"PRAGMA foreign_key_list({t})").fetchall():
            lines.append(f"  {t}.{fk[3]} -> {fk[2]}.{fk[4]}")
    return "\n".join(lines)

CARD = schema_card(con)
print(CARD)

In [ ]:
DICTIONARY = """
### Prairie Outfitters data dictionary (give this to your AI companion as context)
* **stores**: physical stores. `opened_date` is an ISO date.
* **customers**: `loyalty_tier` codes: `B` = Bronze, `S` = Silver, `G` = Gold, `P` = Platinum. Some customers never ordered.
* **products**: `list_price` is the catalog price; `is_active = 0` means discontinued.
* **orders**: one row per order. `channel` is `'online'` or `'in_store'`. **Online orders have `store_id = NULL`.**
  `status` is `'completed'` or `'cancelled'`; cancelled orders never count as revenue. `order_ts` is local time stored as
  text `YYYY-MM-DD HH:MM:SS`.
* **order_items**: one row per product line. The quantity column is **`qty`**. `unit_price` is the price charged before the
  line discount; `discount_pct` is a fraction (0.10 = 10% off).
  **Line revenue** = `qty * unit_price * (1 - discount_pct)`.
* **returns**: one row per returned line; `qty_returned` units, `refund_amount` dollars. Refunds are attributed to the
  period of the *original order*.
* **Net revenue** = line revenue of completed orders minus refunds on those lines.
* Data covers January 2024 through December 2025. The database is **SQLite**.
"""

def extract_sql(text):
    m = re.search(r"```(?:sql|sqlite)?\s*(.*?)```", text, flags=re.S | re.I)
    return (m.group(1) if m else text).strip()

prompt_log = []
def log(part, technique, prompt, sql, result_note, verification, success):
    prompt_log.append(dict(part=part, technique=technique, provider=llm.provider, model=llm.model, prompt=prompt,
                           sql=sql, result=result_note, verification=verification, success=success))

---
# Part B. The context experiment (20 points)

Business question: **What was 2025 net revenue by channel?** (Use the definition in the data dictionary.)

Ask the same question three ways, run each SQL, and fill in the table that follows.
1. **B1, zero-shot, no context:** "Write a SQLite query for Prairie Outfitters' 2025 net revenue by channel."
2. **B2, schema only:** include the schema card.
3. **B3, schema + dictionary:** include the schema card and the data dictionary.

Then write your own **independent verification query** (written or checked by you) and compare all three results with it.
Hint: think about cancelled orders, discounts, refunds, and NULL `store_id`s.

In [ ]:
# B1: zero-shot, no context
b1_prompt = "Write a SQLite query for Prairie Outfitters' 2025 net revenue by channel. Return only SQL."
b1_sql = extract_sql(llm.ask(b1_prompt))
print(b1_sql)
try:
    display(run_sql(b1_sql))
except Exception as exc:
    print("ERROR:", exc)

In [ ]:
# B2: schema only (write the prompt)

In [ ]:
# B3: schema + data dictionary (write the prompt)

In [ ]:
# Your independent verification query (explain in comments why you trust it)

**B4. Results table and discussion.** For B1, B2, and B3 report: did it run, which tables and columns did it invent or
miss, did it exclude cancelled orders, apply discounts, subtract refunds, and does the result match your verification?
Then, in 150 words or fewer: what does this experiment show about context in prompts?

*Your answer here.*

---
# Part C. Six business questions (42 points, 7 each)
For each question: use the assigned technique, show the prompt(s), run the final SQL, perform at least one independent
verification, and add one `log(...)` entry. Keep each question in its own cells.

### C1. What were the top 5 products by units sold in Q2 2025 (April to June, completed orders), with their category? Break ties alphabetically.
**Technique:** Few-shot (house style: one CTE, snake_case aliases, ORDER BY the metric)

In [ ]:
# C1: prompt(s) and final SQL

In [ ]:
# C1: verification (and log(...) entry)

### C2. How many distinct customers placed at least one completed order in each month of 2025?
**Technique:** Chain-of-thought (ask for step-by-step reasoning before the SQL; read the reasoning)

In [ ]:
# C2: prompt(s) and final SQL

In [ ]:
# C2: verification (and log(...) entry)

### C3. What is the return rate (units returned / units sold, completed orders) by product category, for categories with at least 50 units sold?
**Technique:** Chain-of-thought + a fan-out check (compare unit totals before and after your joins)

In [ ]:
# C3: prompt(s) and final SQL

In [ ]:
# C3: verification (and log(...) entry)

### C4. How many customers made their first completed order online and later made a completed in-store purchase?
**Technique:** Tree-of-thought (ask for two approaches, a comparison, and a recommendation)

In [ ]:
# C4: prompt(s) and final SQL

In [ ]:
# C4: verification (and log(...) entry)

### C5. Which store had the highest average order value for completed in-store orders in 2025? Rank all stores.
**Technique:** Self-consistency (two different formulations; compare results programmatically)

In [ ]:
# C5: prompt(s) and final SQL

In [ ]:
# C5: verification (and log(...) entry)

### C6. For each loyalty tier (use the full tier names), what is the average number of days between a customer's consecutive completed orders?
**Technique:** Prompt chaining (plan -> SQL -> senior-engineer review, as three separate calls)

In [ ]:
# C6: prompt(s) and final SQL

In [ ]:
# C6: verification (and log(...) entry)

---
# Part D. Failure log (15 points)
Document at least **three** cases from this assignment where AI-generated SQL was wrong. For each: the prompt, what was
wrong (classify it: hallucinated table or column, wrong dialect, wrong business logic, fan-out or double counting, wrong
filter or date range, other), how you detected it, and how you fixed it (better prompt, more context, manual edit).

In [ ]:
failure_log = [
    # dict(question="B1", prompt="...", problem_type="hallucinated column", what_was_wrong="...", detected_by="...", fix="...")
]
pd.DataFrame(failure_log)

In [ ]:
pd.DataFrame(prompt_log)

---
# Part E. Reflection memo (13 points)
In 250 to 350 words, write a memo to the Prairie Outfitters analytics manager answering: *Under what conditions should
analysts be allowed to report numbers from AI-generated SQL?* Address context (schema, definitions), verification
practices, documentation (prompt logs), and one risk that verification cannot fully remove.

*Your memo here.*

---
# Make your submission PDF (run this last)
Save your work first (**File > Save**), then run the cell below. It turns the **whole** notebook, outputs included,
into `HW1_submission.pdf` and downloads it, plus an HTML copy as a backup. The first run takes one to two minutes because it
installs a PDF printer.

Do not use **File > Print** in Colab: it prints only what is laid out on screen, so long notebooks are cut off partway
down a page. If the PDF step fails, open the downloaded `HW1_submission.html` in your browser and print that page to PDF
(Ctrl+P or Cmd+P, then *Save as PDF*); a standalone page prints in full.

In [ ]:
#@title Make the PDF and HTML copies of this notebook
SUBMISSION_NAME = "HW1_submission"     # file names of the PDF and HTML copies
UPLOAD_INSTEAD = False      # True: convert an uploaded .ipynb instead of this notebook

import json, os, subprocess, sys, textwrap

def _notebook_json():
    """The current notebook, outputs included. Colab first; otherwise ask for the downloaded .ipynb."""
    global SUBMISSION_NAME
    try:
        if UPLOAD_INSTEAD:
            raise LookupError("upload requested")
        from google.colab import _message
        reply = _message.blocking_request("get_ipynb", timeout_sec=120)
        nb = (reply or {}).get("ipynb")
        if nb and nb.get("cells"):
            return nb
        print("Colab did not return the notebook; falling back to an upload.")
    except ImportError:
        pass
    except LookupError:
        pass
    except Exception as exc:
        print(f"Could not read the notebook from Colab ({type(exc).__name__}); falling back to an upload.")
    try:
        from google.colab import files
    except ImportError:
        print("Not running in Colab: in Jupyter, use File > Save and Export Notebook As > PDF or HTML instead.")
        return None
    if UPLOAD_INSTEAD:
        print("Choose the .ipynb file you downloaded from your assignment (File > Download > Download .ipynb).")
    else:
        print("Download this notebook (File > Download > Download .ipynb), then choose that file below.")
    uploaded = files.upload()
    if not uploaded:
        return None
    name, data = next(iter(uploaded.items()))
    if UPLOAD_INSTEAD:
        SUBMISSION_NAME = os.path.splitext(os.path.basename(name))[0].replace(" ", "_")
    return json.loads(data.decode("utf-8"))

PRINT_CSS = """
<style>
  @page { size: Letter; margin: 0.5in; }
  body { background: white !important; }
  .jp-OutputArea-output, .jp-Cell-outputArea, .output_scroll, .jp-OutputArea { overflow: visible !important; max-height: none !important; }
  pre, code { white-space: pre-wrap !important; word-break: break-word; }
  .jp-RenderedHTMLCommon table { font-size: 10px; table-layout: auto; }
  .jp-RenderedHTMLCommon td, .jp-RenderedHTMLCommon th { word-break: break-word; }
  img { max-width: 100% !important; height: auto !important; }
  .jp-Cell { break-inside: auto; }
</style>
"""

def _to_html(nb_dict, path):
    try:
        import nbformat
        from nbconvert import HTMLExporter
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nbconvert"], check=True)
        import nbformat
        from nbconvert import HTMLExporter
    nb_dict.setdefault("nbformat", 4); nb_dict.setdefault("nbformat_minor", 5)
    node = nbformat.reads(json.dumps(nb_dict), as_version=4)   # joins sources and outputs stored as lists of lines
    for cell in node.cells:
        if cell.get("cell_type") == "code" and cell.get("metadata", {}).get("cellView") == "form":
            first = cell["source"].split("\n", 1)[0]   # course-provided form cells (setup, data): title only
            cell["source"] = (first if first.startswith("#@title") else "#@title (setup cell)") + "\n# code hidden in this copy"
    node.metadata.pop("widgets", None)            # widget state makes the page heavy and is not needed on paper
    html, _ = HTMLExporter(template_name="lab").from_notebook_node(node)
    html = html.replace("</head>", PRINT_CSS + "</head>", 1)
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    return path

PRINTER = textwrap.dedent("""
    import sys
    from playwright.sync_api import sync_playwright
    src, out = sys.argv[1], sys.argv[2]
    with sync_playwright() as p:
        browser = p.chromium.launch()
        page = browser.new_page()
        page.goto("file://" + src, wait_until="load", timeout=120000)
        try:
            page.wait_for_load_state("networkidle", timeout=20000)   # lets MathJax finish if a cell uses math
        except Exception:
            pass
        page.emulate_media(media="screen")
        page.pdf(path=out, format="Letter", print_background=True, scale=0.85,
                 margin=dict(top="0.5in", bottom="0.5in", left="0.5in", right="0.5in"))
        browser.close()
""")

def _run(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)

def _to_pdf(html_path, pdf_path):
    try:
        import playwright  # noqa: F401
    except ImportError:
        print("Installing the PDF printer (one time per Colab session)...")
        _run([sys.executable, "-m", "pip", "install", "-q", "playwright"])
    _run([sys.executable, "-m", "playwright", "install", "chromium"])
    with open("_print_pdf.py", "w") as f:
        f.write(PRINTER)
    cmd = [sys.executable, "_print_pdf.py", os.path.abspath(html_path), os.path.abspath(pdf_path)]
    r = _run(cmd)
    if r.returncode != 0:                         # usually missing system libraries on a fresh machine
        _run([sys.executable, "-m", "playwright", "install-deps", "chromium"])
        r = _run(cmd)
    if r.returncode != 0:
        print("PDF step failed:", (r.stderr or r.stdout).strip().splitlines()[-1:])
        return None
    return pdf_path

nb_json = _notebook_json()
if nb_json is not None:
    n_cells = len(nb_json.get("cells", []))
    n_out = sum(1 for c in nb_json.get("cells", []) if c.get("cell_type") == "code" and c.get("outputs"))
    print(f"Notebook read: {n_cells} cells, {n_out} with outputs.")
    if n_out == 0:
        print("Warning: no outputs found. Run your cells (Runtime > Run all) and save before exporting.")
    html_file = _to_html(nb_json, SUBMISSION_NAME + ".html")
    pdf_file = _to_pdf(html_file, SUBMISSION_NAME + ".pdf")
    try:
        from google.colab import files
        for path in [pdf_file, html_file]:
            if path:
                files.download(path)
    except ImportError:
        pass
    if pdf_file:
        print(f"Done: {pdf_file} ({os.path.getsize(pdf_file) // 1024} KB) and {html_file}. "
              "Open the PDF and check the last page before you submit.")
    else:
        print(f"Only the HTML copy was made: open {html_file} in your browser and print it to PDF (Ctrl+P or Cmd+P).")